# 04 - Run the audits

Stage 6: the six audit layers over the 240 unlearned models. **Attach two datasets**: CIFAR-10 and the merged artifacts dataset (stages 3-5 in one place, ~14 GB). The setup cell links it into the store by symlink -- nothing is copied, and the working quota stays free.

**Set `ACCOUNT` and `OF` below.** With every checkpoint in one dataset every account sees all 240 models, so the work is split -- by *condition*, not by model, because each condition pays a setup cost (5 oracles, 5 originals, 32 shadows, 11 relearning anchors) that should be paid once. Three accounts get 3/3/2 conditions. `OF = 1` audits everything on one account.

**Artifacts are frozen from here on.** Stage 6 produces records only -- a few MB of Parquet -- and the upload cell sends *just those* to a small separate dataset, `forgetcheck-records`. The 14 GB is never uploaded again. If a previous audit session already versioned that dataset, attach it too: its records are restored, and models already audited are skipped.

Rough cost: ~2 hours per three-condition share on a T4, dominated by relearning. To get a real number first, add `--forget rand-500` to both audit cells.


In [ ]:
# --- clone the repo at a PINNED commit ------------------------------------------------------
# Clone rather than `pip install git+...`. A wheel would contain only src/forgetcheck/, but the
# CLI also needs configs/ (the metric registry, seed streams, audit protocols) and
# data/memorization/ (the RUM scores -- 400 KB, committed precisely so a fresh session does not
# have to re-download 2 GB from Google Drive).
#
# Pinning is what makes provenance work: every record this session writes carries this commit,
# so any result can be traced back to the exact code that produced it.
REPO   = "https://github.com/hyperreal2005/Minor-Project.git"
COMMIT = "main"          # <-- pin to a sha for real runs, e.g. "a1b2c3d"

import os
from pathlib import Path

os.chdir("/kaggle/working")
if not Path("Minor-Project").exists():
    !git clone --quiet $REPO
%cd /kaggle/working/Minor-Project
!git fetch --quiet --all && git checkout --quiet $COMMIT
!git log -1 --format="pinned at %h  %s"
!pip install -q -e .


In [ ]:
import importlib
import sys
from pathlib import Path

REPO_DIR = Path("/kaggle/working/Minor-Project")

# Make the package importable *in this kernel*.
#
# `pip install -e .` writes a .pth file into site-packages, and .pth files are only processed at
# interpreter startup. The kernel was already running when the previous cell installed, so
# sys.path never picked it up and `import forgetcheck` fails with ModuleNotFoundError. The
# `!forgetcheck` CLI calls below are unaffected -- each spawns a fresh Python that does read the
# .pth -- which makes this failure look stranger than it is.
#
# Adding src/ directly is deterministic and avoids making anyone restart the kernel.
SRC = str(REPO_DIR / "src")
if SRC not in sys.path:
    sys.path.insert(0, SRC)
importlib.invalidate_caches()

# `!` cells run in a subshell, which inherits this. Belt and braces: the console script should
# already be on PATH after the editable install, but if it is not, PYTHONPATH keeps
# `python -m forgetcheck.cli` working as a fallback.
import os
os.environ["PYTHONPATH"] = SRC + os.pathsep + os.environ.get("PYTHONPATH", "")

import forgetcheck
print("forgetcheck imported from:", Path(forgetcheck.__file__).parent)

import shutil

# CIFAR-10 downloads at 100-130 kB/s on Kaggle -- 20 to 30 minutes, repeated on every session and
# every account. Attaching it as a Dataset (Add Input -> Datasets) skips that entirely:
# torchvision checks the md5s of the extracted folder and only downloads if it is missing or
# corrupt. 00_verify_setup.ipynb has a cell that creates the dataset once.
#
# Kaggle's mount layout varies with how a dataset was uploaded -- it may sit at
# /kaggle/input/<slug>/, or nested as /kaggle/input/datasets/<user>/<slug>/, or one level deeper
# again if the dataset was created from a notebook's output directory. So SEARCH for the folder
# rather than assume a path (`**/` matches at any depth, including directly under /kaggle/input),
# and then verify the copy actually landed. An earlier version of this cell
# used `cp ... 2>/dev/null || true` followed by an unconditional success message: a failed copy
# reported success and CIFAR silently re-downloaded anyway, costing ~28 minutes while the output
# claimed otherwise. Never report an outcome that was not checked.
def _restore(name, dest):
    "Find directory `name` anywhere under /kaggle/input and copy it to `dest`."
    dest = Path(dest)
    if dest.is_dir():
        print(f"{name}: already present")
        return True
    found = sorted(Path("/kaggle/input").glob(f"**/{name}"))
    if not found:
        return False
    dest.parent.mkdir(parents=True, exist_ok=True)
    shutil.copytree(found[0], dest, dirs_exist_ok=True)
    print(f"{name}: copied from {found[0]}")
    return True

def _restore_all(name, dest):
    # Merge EVERY directory called `name` under /kaggle/input into `dest`, by symlink.
    #
    # `_restore` above takes the first match, which is right for stages 3-5: each account
    # needs one previous dataset. Stage 6 is different -- an account needs its own Stage 5
    # shard, the Stage 3 oracles and originals, and the Stage 4 shadows, which live in three
    # datasets. Only a merge gives the audit runner all of them under one store root.
    #
    # Symlinks, not copies: /kaggle/input is a read-only mount on a different filesystem, so
    # hard links are impossible and copies would move ~8 GB into the 19.5 GB working quota
    # for nothing. torch.load and Path.is_file() follow symlinks transparently. First match
    # wins for a file present in several datasets (the same run computed once; the pilot's
    # duplicate scrub runs are the only such case), and the count is printed.
    #
    # (Comments, not a docstring: this function lives inside a triple-quoted notebook
    # template, and a nested triple quote ends the template early.)
    dest = Path(dest)
    found = sorted(Path("/kaggle/input").glob(f"**/{name}"))
    found = [f for f in found if f.is_dir()]
    if not found:
        return False
    linked = dup = empty = 0
    empty_example = None
    for src_root in found:
        n_here = 0
        for src in src_root.rglob("*"):
            if not src.is_file():
                continue
            # A zero-byte file in a dataset is almost always a symlink that was zipped as a
            # pointer instead of its target. Link it anyway (the runner recomputes unreadable
            # caches), but say so HERE, at restore time, not thirty failures later.
            if src.stat().st_size == 0:
                empty += 1
                empty_example = empty_example or src
            target = dest / src.relative_to(src_root)
            if target.exists() or target.is_symlink():
                dup += 1
                continue
            target.parent.mkdir(parents=True, exist_ok=True)
            os.symlink(src, target)
            linked += 1
            n_here += 1
        # The mount layout under /kaggle/input is Kaggle's to decide and has changed before.
        # Print the real path rather than letting anyone assume one.
        print(f"{name}: {n_here:5d} files linked from {src_root}")
    print(f"{name}: linked {linked} files from {len(found)} dataset(s)"
          + (f", {dup} already present" if dup else ""))
    if empty:
        print(f"!! {name}: {empty} ZERO-BYTE files, e.g. {empty_example}")
        print("!! That dataset version was uploaded from symlinks, not their targets. Cached")
        print("!! outputs will be recomputed; records would be LOST -- check the .parquet count.")
    return True

# Restore CIFAR-10 by locating its *contents*, not its folder name.
#
# Kaggle does not necessarily preserve the directory that was uploaded: the batches may end up
# inside `cifar-10-batches-py/`, or flattened straight to the dataset root. Searching for the
# folder name therefore reports "not found" while the data sits one level up in plain view --
# which is exactly what happened here, and cost a 25-minute re-download.
#
# So anchor on a file that must exist (`test_batch`) and take whatever directory contains it.
# That handles both layouts, and any future one.
def _restore_cifar(dest):
    dest = Path(dest)
    if dest.is_dir() and len(list(dest.glob("*_batch*"))) == 6:
        print("cifar-10: already present")
        return True
    hits = sorted(Path("/kaggle/input").glob("**/test_batch"))
    if not hits:
        return False
    src = hits[0].parent
    dest.mkdir(parents=True, exist_ok=True)
    for p in src.iterdir():
        if p.is_file():
            shutil.copy2(p, dest / p.name)
    print(f"cifar-10: copied from {src}")
    return True

CIFAR_DEST = REPO_DIR / "data" / "cifar-10-batches-py"
if not _restore_cifar(CIFAR_DEST):
    print("!! no CIFAR-10 batches found under /kaggle/input -- it will DOWNLOAD (~25 min)")
    print("!! attached:", [p.name for p in sorted(Path("/kaggle/input").glob("*"))] or "(none)")

# Verify rather than trust: torchvision needs 5 training batches plus test_batch.
if CIFAR_DEST.is_dir():
    n = len(list(CIFAR_DEST.glob("*_batch*")))
    print(f"   {n}/6 batch files{'' if n == 6 else '  <-- INCOMPLETE, will re-download'}")

# Previous artefacts, so runs another session already finished are skipped rather than repeated.
for _name in ("artifacts", "results"):
    if not _restore_all(_name, REPO_DIR / _name):
        print(f"{_name}: none attached - starting fresh")

import torch
if torch.cuda.is_available():
    print("gpu:", torch.cuda.get_device_name(0))
else:
    print("!! running on CPU. Set Settings -> Accelerator -> GPU.")
    print("!! On CPU one 30-epoch training run takes ~4.8 hours instead of ~12 minutes.")

CLI = "forgetcheck" if shutil.which("forgetcheck") else f"{sys.executable} -m forgetcheck.cli"
print("cli:", CLI)


In [ ]:
ACCOUNT = 1     # <-- this account's number, 1-based
OF      = 3     # <-- how many accounts share the audit (1 takes everything)

DEVICE = "cuda" if __import__("torch").cuda.is_available() else "cpu"
print(f"account {ACCOUNT} of {OF}, device {DEVICE}")


## What was attached, and where Kaggle mounted it

The mount layout under `/kaggle/input` is Kaggle's to decide and has changed before. Nothing in the setup cell assumes it -- every lookup is a recursive search -- but seeing the real paths here is what stops anyone typing one from memory.

In [ ]:
!find /kaggle/input -maxdepth 4 -type d | sort | head -40
!echo; echo 'zero-byte files across ALL inputs (should be 0):'; find /kaggle/input -type f -size -1c | wc -l
!echo 'record shards attached:'; find /kaggle/input -name '*.parquet' | wc -l
!echo 'cached outputs attached:'; find /kaggle/input -name '*.npz' | wc -l


## What is in the store

Every stage should read complete; stale would mean a checkpoint from a superseded method configuration was attached.

In [ ]:
!{CLI} --root . status


## What this account will audit

In [ ]:
!{CLI} --root . --dry-run audit --account {ACCOUNT} --of {OF}


## Run it

One line per model. `skipped:` on a line means a reference was missing for that audit -- see the note at the end of each condition. Models whose audit records already exist are skipped, so a dead session resumes here.

In [ ]:
!{CLI} --root . --device {DEVICE} audit --account {ACCOUNT} --of {OF}


## Inspect what came out

In [ ]:
from forgetcheck.registry import read_records
import pandas as pd

df = read_records("results/records")
aud = df[df["audit"] != "meta"]
print(f"{len(aud)} audit rows across {aud['run_id'].nunique()} models, "
      f"{aud['audit'].nunique()} audits\n")

print("rows per audit / metric:")
print(aud.groupby(["audit", "metric"]).size().to_string(), "\n")

undefined = aud[aud["metric"] == "audit_undefined"]
if len(undefined):
    print(f"{len(undefined)} undefined values (expected on the collapsed neggrad control):")
    print(undefined.groupby(["audit", "probe_set"]).size().to_string(), "\n")

# One headline per family, forget-set probe, averaged over seeds within method x condition.
head = ["js_to_oracle", "mia_auc_pop", "mia_auc_rmia", "cka_linear", "relearn_norm", "sde_margin"]
sub = aud[aud["metric"].isin(head) & aud["probe_set"].isin(["forget", "layer4"])]
wide = sub.pivot_table(index=["forget_id", "method"], columns="metric", values="value")
wide.round(3)


### What to look for

**Every model should have rows from every audit.** The counts table lists six audits; if one is
missing, the reference it needs was not attached -- the run log says which
(`no oracle ensemble in this store` / `no shadow models in this store`) and the fix is to attach
the Stage 3 and Stage 4 artifact datasets, not to re-run.

**`neggrad` rows will include `audit_undefined`.** That is the destroyed control doing its job:
a constant predictor has no per-example signal, so CKA and parts of the privacy attacks are
genuinely undefined. The row records that fact. Do not "fix" it.

**`relearn_norm` on the `original` anchor is not in this table** -- the anchors are computed but
only the method arm is recorded. The protocol check (original ~1, oracle ~0, randinit well below
0) is Stage 7's job and reads the same curves.

**Disagreement is the result.** A method that scores well on `js_to_oracle` and badly on
`mia_auc_rmia`, or well on `cka_linear` and badly on `relearn_norm`, is not an error in either
audit. It is the finding the project exists to produce. Look for it rather than past it.


In [ ]:
from forgetcheck.audits import audit_names

present = set(aud["audit"].unique())
missing = set(audit_names()) - present
print("audits present:", sorted(present))
if missing:
    print("!! audits with NO rows:", sorted(missing), "-- check the run log for 'skipped'")

# Only the unlearned models are targets. Oracles and originals carry relearning-anchor rows
# under their own ids, which is correct and not "missing audits".
targets = aud[aud["role"] == "unlearn"]
per_model = targets.groupby("run_id")["audit"].nunique()
short = per_model[per_model < len(audit_names())]
print(f"models with all {len(audit_names())} audits: {(per_model == len(audit_names())).sum()}"
      f" / {len(per_model)}")
if len(short):
    print("!! models missing audits:", len(short), "-- e.g.", short.index[0])

if "audit_undefined" in aud["metric"].values:
    who = aud[aud["metric"] == "audit_undefined"]["method"].value_counts()
    print("undefined values by method:", who.to_dict(),
          "<-- expected: neggrad; unexpected: anything else")


In [ ]:
# --- push Stage 6's outputs -----------------------------------------------------------------
# Stage 6 writes three things, none of them checkpoints:
#   results/                -- the audit records (a few MB)
#   artifacts/outputs/      -- each model's logits on the probe sets (~0.4 MB each)
#   artifacts/activations/  -- GAP-pooled activations, fp16 (~6 MB each)
# The checkpoints (14 GB) are frozen after Stage 5 and never uploaded again. These three go to
# a SEPARATE dataset, ~2 GB at most, and with them every audit except relearning can be re-run
# on a laptop without a GPU -- a changed bandwidth or binning costs seconds, not hours.
#
# Copied with symlinks dereferenced (copytree's default): restored files are symlinks into
# /kaggle/input, and a zip of symlinks would upload pointers, not data.
#
# `kaggle datasets version` is a full snapshot, so each upload must contain everything so far
# -- which it does, because the setup cell restored the previous version first. Run the three
# accounts' uploads one after another, each attaching the latest version.
import json, shutil
from pathlib import Path

OUT = Path("/kaggle/working/to_upload")
if OUT.exists():
    shutil.rmtree(OUT)
OUT.mkdir(parents=True)
shutil.copytree(REPO_DIR / "results", OUT / "results", symlinks=False)
for kind in ("outputs", "activations"):
    src = REPO_DIR / "artifacts" / kind
    if src.is_dir():
        shutil.copytree(src, OUT / "artifacts" / kind, symlinks=False)
shards = sorted((OUT / "results").rglob("*--audit-*.parquet"))
n_out = sum(1 for p in OUT.rglob("*.npz"))
mb = sum(p.stat().st_size for p in OUT.rglob("*") if p.is_file()) / 1e6
print(f"staged {len(shards)} audit shards, {n_out} cached output files, {mb:.0f} MB")
if not shards:
    raise SystemExit("no audit records to upload; did the audit cell run?")

(OUT / "dataset-metadata.json").write_text(json.dumps({
    "title": "forgetcheck-stage6",
    "id": "YOUR-KAGGLE-USERNAME/forgetcheck-stage6",   # <-- records + cached outputs, ~2 GB
    "licenses": [{"name": "CC0-1.0"}],
}, indent=2))
# First time:   !kaggle datasets create  -p /kaggle/working/to_upload --dir-mode zip
# Afterwards:   !kaggle datasets version -p /kaggle/working/to_upload -m "stage 6 account K" --dir-mode zip
